# Netflix Hit Proposal

Working notebook for the capstone deliverable: use real data from `netflix/data/*.csv` to (1) define what counts as a "hit", (2) find which pre-release-knowable features correlate with it, and (3) pitch an original series concept backed by that evidence.

This notebook runs on the **real** composite datasets (not the synthetic stand-in used in `netflix_review_concepts_capstone.ipynb`).

In [15]:
import numpy as np
import pandas as pd

netflix = pd.read_csv("netflix/data/netflix.titles.composite.csv")
netflix.shape

(691, 13)

## 1. Data completeness

Before defining anything, check how much of the composite dataset actually has each signal. `netflix_viewing_hours` and `netflix_weeks` come straight from Netflix's own Top 10 data and are complete for all 691 titles. The `tmdb_*` and `imdb_*` columns depend on a fuzzy-match join and are missing for a meaningful share of rows -- this is why they should stay out of the `hit` label itself (see Section 2).

In [16]:
missing_pct = netflix.isna().mean().mul(100).round(1).rename("missing_pct")
missing_pct.to_frame()

,missing_pct
key,0.0
netflix_viewing_hours,0.0
netflix_weeks,0.0
netflix_year_hint,0.0
netflix_title,0.0
netflix_clean_title,0.1
tmdb_title,43.0
tmdb_popularity,43.0
tmdb_vote_average,43.0
tmdb_vote_count,43.0


## 2. Do viewing hours, votes, and ratings actually move together?

Log1p-transform the heavily skewed count-like columns (`viewing_hours`, `weeks`, `tmdb_popularity`, `tmdb_vote_count`, `imdb_numVotes`) before correlating -- their raw distributions span orders of magnitude and would otherwise be dominated by a handful of outliers (e.g. Stranger Things).

**Finding:** `netflix_viewing_hours` correlates strongly with `netflix_weeks` (0.81) but only weakly with `imdb_averageRating` (0.18) and barely at all with `imdb_numVotes` (0.08). Reach (how much something got watched) and reception (how well it was rated) are largely independent signals here -- which is exactly why they shouldn't be collapsed into one `hit` label. `viewing_hours` and `weeks` are the only two that clearly measure the same underlying thing (Netflix-internal popularity), so those are the two combined into `hit_score` below.

In [17]:
log_cols = {
    "viewing_hours": np.log1p(netflix["netflix_viewing_hours"]),
    "weeks": np.log1p(netflix["netflix_weeks"]),
    "tmdb_popularity": np.log1p(netflix["tmdb_popularity"]),
    "tmdb_vote_average": netflix["tmdb_vote_average"],
    "tmdb_vote_count": np.log1p(netflix["tmdb_vote_count"]),
    "imdb_averageRating": netflix["imdb_averageRating"],
    "imdb_numVotes": np.log1p(netflix["imdb_numVotes"]),
}
corr_df = pd.DataFrame(log_cols)
corr_df.corr().round(2)

,viewing_hours,weeks,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,imdb_averageRating,imdb_numVotes
viewing_hours,1.00,0.81,0.39,0.27,0.44,0.18,0.08
weeks,0.81,1.00,0.27,0.13,0.25,0.14,0.07
tmdb_popularity,0.39,0.27,1.00,0.66,0.87,0.31,0.33
tmdb_vote_average,0.27,0.13,0.66,1.00,0.76,0.20,0.25
tmdb_vote_count,0.44,0.25,0.87,0.76,1.00,0.28,0.39
imdb_averageRating,0.18,0.14,0.31,0.20,0.28,1.00,0.02
imdb_numVotes,0.08,0.07,0.33,0.25,0.39,0.02,1.00


## 3. Defining `hit_score`

Decision: combine `netflix_viewing_hours` (total exposure) and `netflix_weeks` (durability -- did it stick around or spike and vanish) into one composite target. Ratings and vote counts stay out: they're weakly correlated with viewing hours (Section 2), would drop the usable sample from 691 to as few as 278 rows, and -- for the proposal use case -- don't exist yet for an unmade show anyway.

**Combination method:** z-score standardize each of `log1p(viewing_hours)` and `log1p(weeks)` independently, average them with equal weight, then min-max the result to `[0, 1]`.

Why equal weight and not a fitted weight: with exactly two already-standardized variables, PCA's first component is *always* the equal-weighted average, regardless of their correlation (the eigenvectors of `[[1, rho], [rho, 1]]` are always `(1,1)` and `(1,-1)`). There's also no ground-truth "hit" label available to fit a weight against via regression -- that would be circular, since `hit_score` is the thing being constructed. So equal weight isn't a shortcut; given two standardized inputs and no external criterion, it's what the data-driven approach reduces to.

In [18]:
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

def minmax(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

z_hours = zscore(np.log1p(netflix["netflix_viewing_hours"]))
z_weeks = zscore(np.log1p(netflix["netflix_weeks"]))

netflix["hit_score"] = minmax((z_hours + z_weeks) / 2)
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score"]].sort_values(
    "hit_score", ascending=False
).head(10)

,netflix_title,netflix_viewing_hours,netflix_weeks,hit_score
396,Squid Game,2289500000,20,1.000000
103,Café con aroma de mujer,793740000,25,0.965821
661,Stranger Things,2874260000,13,0.937541
140,"Yo soy Betty, la fea",297560000,30,0.929910
549,Manifest,1446260000,16,0.926189
447,Money Heist,1170200000,14,0.886773
376,The Queen of Flow,561230000,16,0.858605
607,Ozark,751600000,13,0.841773
167,Bridgerton,1108800000,11,0.839614
598,Red Notice,453990000,14,0.819171


## 4. Sanity check

`hit_score` should reward both massive-but-brief spikes and modest-but-sustained runs, not just raw hours. Compare the ranking above to a plain `viewing_hours`-only ranking -- titles with high weeks but comparatively lower hours should move up; one-off spikes with very few weeks should move down relative to the hours-only ranking.

In [19]:
hours_rank = netflix["netflix_viewing_hours"].rank(ascending=False)
hit_rank = netflix["hit_score"].rank(ascending=False)
netflix["rank_shift"] = hours_rank - hit_rank  # positive = moved up under hit_score
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score", "rank_shift"]].sort_values(
    "rank_shift", ascending=False
).head(10)

,netflix_title,netflix_viewing_hours,netflix_weeks,hit_score,rank_shift
128,Beauty and the Beast,7480000,4,0.312790,263.5
377,Shiny_Flakes: The Teenage Drug Lord,10470000,4,0.336799,215.0
422,Grudge,13230000,5,0.388892,206.0
688,Xtreme,5390000,3,0.246083,206.0
180,Lost Bullet,6250000,3,0.256652,199.0
227,The Claus Family,21600000,6,0.453811,192.0
321,El infierno,13520000,4,0.355052,168.0
237,Confessions of an Invisible Girl,19730000,5,0.417426,166.0
569,Meenakshi Sundareshwar,9130000,3,0.283710,160.0
328,أصحاب ...ولا أعزّ,8730000,3,0.280512,156.0


## 5. Building the analysis dataset

Missing-value strategy: **complete-case analysis** -- keep only titles matched across all three sources (`netflix_title` always present, plus non-null `tmdb_title` and `imdb_title`). That's 391 of 691 rows.

**Assumption, stated explicitly:** we assume the upstream fuzzy-match pipeline's `tmdb_title`/`imdb_title` matches are correct. A quick spot-check did turn up some mismatches on numeric-heavy titles (e.g. `1917` matched to TMDB's `1916`) -- for a teaching exercise under a one-night deadline, fixing the matching pipeline itself is out of scope, so this is a known, documented limitation rather than something silently ignored.

Genre, language, and per-episode runtime come from `tmdb.titles.v3.csv`, joined on `tmdb_title == name`. Because some matched titles share a name with another, unrelated TMDB entry, ties are broken by keeping the highest-`vote_count` row for each name -- the more plausible candidate, though not guaranteed correct.

In [20]:
tmdb = pd.read_csv("netflix/data/tmdb.titles.v3.csv")

# complete-case: keep only titles matched in both tmdb and imdb
dataset = netflix.dropna(subset=["tmdb_title", "imdb_title"]).copy()

# de-dupe tmdb by name, keeping the highest-vote_count row for ambiguous names.
# stable mergesort + id as a secondary tiebreaker: sort_values' default quicksort is NOT
# stable, so ties (including NaN vote_count ties) could otherwise pick a different row on
# every run, silently changing every downstream number.
tmdb_features = (
    tmdb.sort_values(["vote_count", "id"], ascending=[False, True], kind="mergesort")
    .drop_duplicates(subset="name", keep="first")[["name", "genres", "original_language", "episode_run_time"]]
)

dataset = dataset.merge(tmdb_features, left_on="tmdb_title", right_on="name", how="left")
dataset = dataset.drop(columns="name")

print(f"analysis dataset: {dataset.shape[0]} rows (from {len(netflix)} total, {len(netflix) - dataset.shape[0]} dropped for missing tmdb/imdb match)")
dataset[["netflix_title", "hit_score", "genres", "original_language", "episode_run_time"]].head(10)

analysis dataset: 391 rows (from 691 total, 300 dropped for missing tmdb/imdb match)


,netflix_title,hit_score,genres,original_language,episode_run_time
0,The 100,0.304167,"Sci-Fi & Fantasy, Drama, Action & Adventure",en,43
1,1000 Miles from Christmas,0.256272,Documentary,en,42
2,Love 101,0.172172,"Comedy, Drama",tr,45
3,Back to 15,0.197875,"Comedy, Drama",pt,40
4,1917,0.161430,Documentary,en,83
5,Formula 1: Drive to Survive,0.358672,Documentary,en,38
6,Blade Runner 2049,0.138388,Sci-Fi & Fantasy,en,0
7,211,0.138842,Documentary,es,0
8,21 Jump Street,0.166820,"Crime, Mystery, Drama",en,45
9,Monsters Inside: The 24 Faces of Billy Milligan,0.185331,"Documentary, Crime",en,60


## 6. Encoding features for regression

Three raw columns, three different encoding problems:

- **`genres`** is multi-label (a title can be `"Drama, Crime, Mystery"`), so it needs multi-hot encoding -- one binary column per genre, not a single categorical. Rare genres (`News`, `Western`, `Talk`, `War & Politics`, each under 10 titles) are dropped rather than encoded: with only 334 rows, a dummy that's `1` for two or three titles gives linear regression almost nothing to estimate a stable coefficient from. Titles left with none of the kept genres simply have all genre columns at `0` -- they fall into the same implicit baseline as the reference category dropped by `original_language`'s one-hot encoding below.
- **`original_language`** is a plain categorical, but has a long tail (24 languages, many appearing 1-3 times). Same reasoning as above: bucket anything under 10 titles into `other`, then one-hot with `drop_first=True` (English becomes the implicit baseline).
- **`episode_run_time`** has 55 rows (16%) sitting at exactly `0`, which isn't a plausible episode length -- almost certainly unpopulated TMDB data rather than a true zero. Treated as missing: replaced with the column median and flagged with a `runtime_missing` indicator, so the model can still tell those rows apart from titles with a genuinely short runtime.

In [21]:
# drop the rows where tmdb matched but had no genre data of its own
dataset = dataset.dropna(subset=["genres"]).reset_index(drop=True)

# --- genres: multi-hot, keep only genres with >= 10 occurrences ---
genre_lists = dataset["genres"].str.split(", ")
genre_counts = genre_lists.explode().str.strip().value_counts()
keep_genres = genre_counts[genre_counts >= 10].index.tolist()
for g in keep_genres:
    dataset[f"genre_{g}"] = genre_lists.apply(lambda lst: g in [x.strip() for x in lst]).astype(int)

# --- original_language: bucket rare languages, then one-hot ---
lang_counts = dataset["original_language"].value_counts()
keep_langs = lang_counts[lang_counts >= 10].index.tolist()
dataset["language_bucketed"] = dataset["original_language"].where(
    dataset["original_language"].isin(keep_langs), "other"
)
lang_dummies = pd.get_dummies(dataset["language_bucketed"], prefix="lang", drop_first=True).astype(int)
dataset = pd.concat([dataset, lang_dummies], axis=1)

# --- episode_run_time: treat 0 as missing, impute with median, flag it ---
dataset["runtime_missing"] = (dataset["episode_run_time"] == 0).astype(int)
runtime_clean = dataset["episode_run_time"].replace(0, np.nan)
dataset["episode_run_time_imputed"] = runtime_clean.fillna(runtime_clean.median())

feature_cols = (
    [f"genre_{g}" for g in keep_genres]
    + list(lang_dummies.columns)
    + ["episode_run_time_imputed", "runtime_missing"]
)
print(f"final dataset: {dataset.shape[0]} rows, {len(feature_cols)} features")
dataset[["netflix_title", "hit_score", *feature_cols]].head(5)

final dataset: 334 rows, 20 features


,netflix_title,hit_score,genre_Drama,genre_Crime,genre_Comedy,genre_Sci-Fi & Fantasy,genre_Action & Adventure,genre_Mystery,genre_Documentary,genre_Animation,...,genre_Soap,genre_Kids,lang_es,lang_fr,lang_ja,lang_ko,lang_other,lang_pt,episode_run_time_imputed,runtime_missing
0,The 100,0.304167,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,43.0,0
1,1000 Miles from Christmas,0.256272,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,42.0,0
2,Love 101,0.172172,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,45.0,0
3,Back to 15,0.197875,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,40.0,0
4,1917,0.161430,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,83.0,0


## 7. Correlation check before fitting

Two things worth checking before regression: (1) does each feature actually relate to `hit_score` at all, and (2) are any features so correlated with each other that the regression coefficients will be unstable (multicollinearity)?

In [22]:
target_corr = (
    dataset[feature_cols + ["hit_score"]].corr()["hit_score"].drop("hit_score").sort_values(key=abs, ascending=False)
)
print("--- correlation with hit_score ---")
print(target_corr.round(3))

fcorr = dataset[feature_cols].corr()
high_pairs = [
    (a, b, round(fcorr.loc[a, b], 3))
    for i, a in enumerate(feature_cols)
    for b in feature_cols[i + 1 :]
    if abs(fcorr.loc[a, b]) > 0.5
]
print()
print("--- feature pairs with |r| > 0.5 (multicollinearity risk) ---")
print(high_pairs if high_pairs else "none")

--- correlation with hit_score ---
genre_Drama                 0.295
lang_ko                     0.215
lang_other                 -0.156
genre_Documentary          -0.129
genre_Mystery               0.111
genre_Reality              -0.102
lang_fr                    -0.096
genre_Animation            -0.090
episode_run_time_imputed    0.088
genre_Family               -0.077
runtime_missing            -0.071
lang_es                     0.067
genre_Kids                 -0.059
lang_ja                    -0.057
genre_Crime                 0.053
lang_pt                    -0.048
genre_Soap                  0.044
genre_Action & Adventure    0.034
genre_Sci-Fi & Fantasy      0.010
genre_Comedy               -0.009
Name: hit_score, dtype: float64

--- feature pairs with |r| > 0.5 (multicollinearity risk) ---
none


**Findings:** no feature pair exceeds `|r| > 0.5`, so multicollinearity isn't a concern for the regression. Correlation with `hit_score` is weak across the board -- unsurprising, since genre/language/runtime alone are a coarse description of a show -- but `genre_Drama` (0.295) and `lang_ko` (0.215, Korean-language titles) stand out as the two features most worth watching in the regression coefficients.

## 8. Fitting the regression

Same normal-equations approach as Module 2 (`beta = (X'X)^-1 X'y`), extended with standard errors, t-stats, and p-values so coefficients can be judged for statistical significance, not just sign and magnitude -- `numpy` + `scipy.stats.t`, no new library needed.

In [23]:
from scipy import stats

X_raw = dataset[feature_cols].to_numpy(dtype=float)
y = dataset["hit_score"].to_numpy(dtype=float)
n, k = X_raw.shape
X = np.column_stack([np.ones(n), X_raw])
p = X.shape[1]

XtX_inv = np.linalg.inv(X.T @ X)
beta = XtX_inv @ X.T @ y
resid = y - X @ beta
rss = np.sum(resid ** 2)
sigma2 = rss / (n - p)
se = np.sqrt(np.diag(sigma2 * XtX_inv))
t_stats = beta / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - p))

tss = np.sum((y - y.mean()) ** 2)
r2 = 1 - rss / tss
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p)

regression_result = pd.DataFrame(
    {"coef": beta, "se": se, "t": t_stats, "p_value": p_values},
    index=["intercept", *feature_cols],
).sort_values("p_value")

print(f"n={n}, R2={r2:.4f}, adj_R2={adj_r2:.4f}")
regression_result.round(4)

n=334, R2=0.1655, adj_R2=0.1122


,coef,se,t,p_value
intercept,0.3083,0.0475,6.4940,0.0000
lang_other,-0.1161,0.0329,-3.5293,0.0005
genre_Drama,0.1101,0.0336,3.2761,0.0012
lang_fr,-0.1128,0.0596,-1.8937,0.0592
lang_ko,0.0708,0.0397,1.7819,0.0757
lang_pt,-0.0709,0.0620,-1.1429,0.2539
lang_ja,-0.0548,0.0497,-1.1022,0.2712
genre_Family,-0.0408,0.0504,-0.8098,0.4187
genre_Soap,0.0413,0.0611,0.6757,0.4997
genre_Mystery,0.0196,0.0312,0.6300,0.5291


**Results:** R² = 0.166, adjusted R² = 0.112 -- low, as expected, since `genre`/`original_language`/`episode_run_time` are a coarse sketch of a show, not a full description of its writing or marketing. Still, two coefficients clear p < 0.05:

- **`lang_other` (niche/long-tail language): -0.116 (p = 0.0005)** -- titles in a language outside the six most common (en/es/ko/ja/fr/pt) score meaningfully lower, relative to the English baseline. This is the single most significant result in the regression (smallest p-value, largest coefficient magnitude), though it's a "what to avoid" finding rather than something a proposal can act on directly (the show being pitched already commits to a specific major language).
- **`genre_Drama`: +0.110 (p = 0.0012)** -- holding language and runtime fixed, being tagged Drama is associated with a meaningfully higher `hit_score`. Consistent with `genre_Drama`'s 0.295 correlation from Section 7, and the strongest *positive*, actionable result in the regression.

Two more are borderline (0.05 < p < 0.10) and point in *opposite* directions -- worth naming as directional evidence in the proposal, but neither is strong enough to lean on alone: `lang_ko` is **+0.071** (p = 0.076, Korean-language titles trending higher), while `lang_fr` is **-0.113** (p = 0.059, French-language titles trending lower). Everything else (most genres, `episode_run_time`, `runtime_missing`) is statistically indistinguishable from zero at this sample size -- absence of evidence, not evidence that they don't matter; a coarse 334-row, pre-release-only feature set simply can't resolve a subtler effect than that.

## 9. Drilling into the Drama + Korean subset

Rather than adding more weak, redundant whole-dataset predictors (tried `origin_country` and `type` above -- neither held up), dig into the two features that *were* significant. Both `genre_Drama` and `lang_ko` point the same direction, so look at what else the Korean-drama titles have in common -- material to cite directly in the proposal.

In [24]:
dataset["is_drama"] = dataset["genres"].str.contains("Drama")
dataset["is_korean"] = dataset["original_language"] == "ko"

print("mean hit_score by group:")
print(f"  overall:          {dataset['hit_score'].mean():.3f}  (n={len(dataset)})")
print(f"  Drama:            {dataset.loc[dataset.is_drama, 'hit_score'].mean():.3f}  (n={dataset.is_drama.sum()})")
print(f"  Korean:           {dataset.loc[dataset.is_korean, 'hit_score'].mean():.3f}  (n={dataset.is_korean.sum()})")
print(f"  Korean + Drama:   {dataset.loc[dataset.is_drama & dataset.is_korean, 'hit_score'].mean():.3f}  "
      f"(n={(dataset.is_drama & dataset.is_korean).sum()})")

kdrama = dataset[dataset.is_korean].copy()
print()
print("secondary genre, within Korean titles only (small n -- directional, not conclusive):")
for g in ["Comedy", "Action & Adventure", "Mystery", "Sci-Fi & Fantasy", "Crime", "Family"]:
    has = kdrama["genres"].str.contains(g)
    print(
        f"  {g:20s} present: n={has.sum():2d} mean_hit={kdrama.loc[has, 'hit_score'].mean():.3f}   "
        f"absent: n={(~has).sum():2d} mean_hit={kdrama.loc[~has, 'hit_score'].mean():.3f}"
    )

mean hit_score by group:
  overall:          0.363  (n=334)
  Drama:            0.412  (n=193)
  Korean:           0.498  (n=30)
  Korean + Drama:   0.500  (n=29)

secondary genre, within Korean titles only (small n -- directional, not conclusive):
  Comedy               present: n= 7 mean_hit=0.481   absent: n=23 mean_hit=0.503
  Action & Adventure   present: n= 5 mean_hit=0.693   absent: n=25 mean_hit=0.459
  Mystery              present: n= 6 mean_hit=0.576   absent: n=24 mean_hit=0.478
  Sci-Fi & Fantasy     present: n= 5 mean_hit=0.468   absent: n=25 mean_hit=0.504
  Crime                present: n= 4 mean_hit=0.487   absent: n=26 mean_hit=0.499
  Family               present: n= 2 mean_hit=0.267   absent: n=28 mean_hit=0.514


In [25]:
rt_korean = kdrama.loc[kdrama["episode_run_time"] > 0, "episode_run_time"]
rt_other_drama = dataset.loc[
    dataset["genres"].str.contains("Drama") & ~dataset.is_korean, "episode_run_time"
]
rt_other_drama = rt_other_drama[rt_other_drama > 0]

print("episode runtime, Korean titles (0/missing excluded):")
print(rt_korean.describe().round(1))
print()
print("episode runtime, non-Korean Drama titles (0/missing excluded):")
print(rt_other_drama.describe().round(1))

episode runtime, Korean titles (0/missing excluded):
count    26.0
mean     61.9
std      14.9
min      16.0
25%      60.0
50%      63.0
75%      70.0
max      93.0
Name: episode_run_time, dtype: float64

episode runtime, non-Korean Drama titles (0/missing excluded):
count    142.0
mean      47.9
std       20.5
min       10.0
25%       42.0
50%       45.0
75%       52.0
max      200.0
Name: episode_run_time, dtype: float64


**Findings to carry into the proposal:**

- **Korean + Drama is the strongest combination found anywhere in this analysis.** Mean `hit_score` jumps from 0.363 (overall) to 0.500 for the 29 titles that are both -- almost identical to the Korean-only mean (0.498, n=30), because 29 of the 30 Korean titles in this dataset are already tagged Drama. Being Korean-language on Netflix essentially *is* being Korean-drama.
- **Within Korean titles, adding an Action & Adventure tag associates with a noticeably higher mean hit_score** (0.693 vs 0.459 without it) -- direction consistent with *Squid Game* and *All of Us Are Dead* topping the list above. Caveat worth stating plainly in the proposal: only 5 of 30 titles carry that tag, so this is a suggestive pattern, not a statistically resolved effect.
- **Family-tagged Korean titles underperform sharply** (0.267 vs 0.514, n=2) -- again too small a cell to lean on hard, but worth naming as a combination to avoid.
- **Korean dramas run distinctly longer per episode** -- median 63 min (n=26 with real runtime data) vs. 45 min for non-Korean dramas (n=142). This is a structural, format-level difference, not just a genre label, and it's something a proposal can act on directly: pace the pitch for an extended-format episode, not a standard 30-45 minute slot.

## 10. Creative team track record (qualitative, not regression)

Quantifying cast/crew historical performance as a regression feature was tried and dropped: re-matching titles back into `imdb.titles.composite.csv` by title string, restricted to `tvSeries` and case-insensitive, still leaves 30% of the 334-row dataset with zero candidate matches and another 18% ambiguous (multiple candidates needing a tiebreak) -- too unreliable to build a whole-dataset feature on tonight.

Instead: take the 5 Korean titles carrying the `Action & Adventure` tag (Section 9's strongest subgroup) as a small population, and look up their director/writer credits directly -- each title checked individually against `imdb.titles.composite.csv` (unambiguous single match, verified by genre and vote count), rather than an automated bulk join. The premise this checks: real hit-making directors/writers get repeat, better-resourced work -- which is exactly the kind of claim a proposal can point to.

In [26]:
imdb_full = pd.read_csv("netflix/data/imdb.titles.composite.csv")

action_korean_titles = {
    "Squid Game": "tt10919420",
    "All of Us Are Dead": "tt14169960",
    "Alchemy of Souls": "tt20859920",
    "My Name": "tt12940504",
    "Money Heist: Korea": "tt13696452",
}

def credits_person(field: pd.Series, person: str) -> pd.Series:
    return field.fillna("").apply(lambda s: person in s.split(","))

for name, tconst in action_korean_titles.items():
    row = imdb_full.loc[imdb_full["tconst"] == tconst].iloc[0]
    key_people = set(str(row["directors"]).split(",")) | set(str(row["writers"]).split(","))
    key_people.discard("\\N")

    mask = pd.Series(False, index=imdb_full.index)
    for p in key_people:
        mask |= credits_person(imdb_full["directors"], p) | credits_person(imdb_full["writers"], p)
    other_credits = imdb_full[mask].drop_duplicates(subset="tconst")

    print(f"=== {name} ({row.startYear}, rating={row.averageRating}, votes={int(row.numVotes)}) ===")
    print(f"director/writer credited on {len(other_credits)} titles total in the IMDb dataset, top by votes:")
    print(
        other_credits[["primaryTitle", "startYear", "averageRating", "numVotes"]]
        .sort_values("numVotes", ascending=False)
        .head(4)
        .to_string(index=False)
    )
    print()

=== Squid Game (2021, rating=7.9, votes=767607) ===
director/writer credited on 6 titles total in the IMDb dataset, top by votes:
               primaryTitle startYear  averageRating  numVotes
                 Squid Game      2021            7.9  767607.0
      Surreal Entertainment      2018            7.4      58.0
                 The Dealer        \N            NaN       NaN
The Best Show on the Planet        \N            NaN       NaN

=== All of Us Are Dead (2022, rating=7.6, votes=92709) ===
director/writer credited on 13 titles total in the IMDb dataset, top by votes:
          primaryTitle startYear  averageRating  numVotes
    All of Us Are Dead      2022            7.6   92709.0
Daily Dose of Sunshine      2023            8.1    6226.0
          Pyramid Game      2024            7.5    3321.0
     The King 2 Hearts      2012            7.6    1688.0

=== Alchemy of Souls (2022, rating=8.7, votes=29759) ===
director/writer credited on 22 titles total in the IMDb dataset, top

**Findings:** every one of the 5 directors/writers checked has multiple other credited titles, several of them substantial hits in their own right (*Hotel Del Luna*, *What's Wrong with Secretary Kim* for the *Alchemy of Souls* team; *Extracurricular* for the *My Name* director). Most notable: *Money Heist: Korea*'s writing credits trace back to **Álex Pina**, creator of the original Spanish *Money Heist* (612,436 votes -- by far the single most-voted title surfaced anywhere in this whole analysis) -- a direct, verifiable link between a proven global hit-maker and one of this dataset's Korean action-dramas.

This is qualitative, not a regression coefficient -- it's not claiming "more prior credits causes a higher hit_score" (that's exactly the `cast_count` correlation the original module notebook found to be weak, -0.077). What it supports instead is a narrower, defensible claim for a proposal: **the specific people behind this subgroup's biggest successes are established, repeatedly-hired professionals, not one-off flukes** -- useful context for naming a real creative team analogue when pitching.

## 11. Polti's 36 Dramatic Situations -- narrative structure as the 6th feature

Source: [Georges Polti's *The Thirty-Six Dramatic Situations*](https://en.wikipedia.org/wiki/The_Thirty-Six_Dramatic_Situations), a taxonomy claiming every dramatic scenario in storytelling reduces to one of 36 recurring conflict patterns. The five data-driven features above (Korean-language, Drama, Action & Adventure, ~60-70 min episodes, established-creator comps) describe a show's *form*. Polti's framework supplies the 6th: its narrative *structure*.

Two situations map directly onto the Action & Adventure-tagged Korean titles from Section 9:

- **#7 -- Falling Prey to Cruelty**: an Unfortunate is placed at the mercy of an overwhelming, merciless power. *Squid Game*'s contestants (crushed first by economic desperation, then trapped in a game run by anonymous elites) and *All of Us Are Dead*'s students (hunted by an outbreak, then abandoned by the adult institutions meant to protect them) both run on a *double* layer of this situation -- an immediate physical threat plus a structural/institutional one underneath it.
- **#20 -- Self-Sacrifice for an Ideal**: a character gives up self-interest for a principle larger than survival. Both shows repeatedly stage this as the counterweight to #7: characters who could act purely on self-preservation instead choose solidarity, protection of others, or (in *Squid Game*'s case) actively working to dismantle the system that victimized them.

The pairing looks like it isn't incidental: #7 supplies the external stakes and spectacle (the visceral, Action & Adventure-coded half of the pitch), #20 supplies the emotional/moral payoff that turns a survival-game premise into something people discuss and rewatch (the Drama-coded half). That two-situation combination is a plausible structural reading of *why* the Korean action-drama subgroup scores as well as it does in Section 9 -- not just genre tags in isolation, but a specific narrative engine underneath them.

*(Deliberately stopping here -- picking one specific situation combination and writing the actual pitch is the next, separate step, left to the proposal itself rather than this notebook.)*

## Summary: six features for the proposal

Feature selection is done as of here. Six load-bearing pieces of evidence, mixing data-driven (1-4) and qualitative (5-6):

1. **Korean-language** -- `lang_ko` +0.071 coefficient (p=0.076, borderline), and Korean titles average 0.498 `hit_score` vs 0.363 overall (Section 9)
2. **Drama genre** -- `genre_Drama` +0.110 coefficient (p=0.0012, the strongest *positive*, actionable significant result -- `lang_other` is technically more significant but is a "what to avoid" finding, not a pitch-able feature)
3. **Action & Adventure secondary genre** -- within Korean titles, 0.693 vs 0.459 mean `hit_score` (n=5, directional only, Section 9)
4. **Extended episode format (~60-70 min)** -- Korean dramas run a median 63 min/episode vs 45 min for non-Korean dramas (Section 9), a structural difference a pitch can act on directly
5. **Established creative team precedent** -- qualitative, not regression: the directors/writers behind this subgroup's top performers are repeatedly-hired professionals with other substantial hits, including a direct line to *Money Heist*'s creator (Section 10)
6. **Narrative structure via Polti** -- situations #7 (Falling Prey to Cruelty) + #20 (Self-Sacrifice for an Ideal) as the structural engine underneath the top-scoring titles (Section 11)

Writing the actual pitch from here is a separate, subsequent step.